# Set up

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[1]))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from abc import ABC, abstractmethod

In [3]:
from utils import *

## Constants

In [4]:
SEED = 42
LEARNING_RATE = 0.01
MAX_ITER = 10000

# Process data

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. ĐỌC DỮ LIỆU TỪ FILE CSV ĐÃ LƯU
try:
    df = pd.read_csv('data/dataset.csv')
    print("Đã tải dữ liệu thành công.")
except FileNotFoundError:
    print("Lỗi: Không tìm thấy file 'dataset.csv'. Hãy chạy file xử lý trước đó.")
    exit()

# 2. TÁCH BIẾN ĐẶC TRƯNG (X) VÀ BIẾN MỤC TIÊU (y)
# Loại bỏ cột 'Class' (chuỗi) và 'Class_Encoded' (mục tiêu) ra khỏi X
X = df.drop(['Class', 'Class_Encoded'], axis=1)
y = df['Class_Encoded']

# 3. CHIA DỮ LIỆU: TRAIN (70%) / VAL (10%) / TEST (20%)
# Bước 1: Tách 20% cho Test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Bước 2: Tách 10% (trên tổng số) cho Validation từ phần 80% còn lại
# Tỷ lệ test_size = 0.125 vì 0.125 * 0.8 = 0.1 (tức 10%)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.125, random_state=42, stratify=y_train_val
)

# 4. CHUẨN HÓA DỮ LIỆU (STANDARDIZATION)
# Khởi tạo bộ chuẩn hóa
scaler = StandardScaler()

# LƯU Ý: Chỉ "fit" trên tập Train để tránh rò rỉ dữ liệu (Data Leakage)
X_train_scaled = scaler.fit_transform(X_train)

# "transform" cho tập Val và Test dựa trên thông số của tập Train
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 5. KIỂM TRA KẾT QUẢ
print("-" * 30)
print(f"Tổng số mẫu: {len(df)}")
print(f"Tập Huấn luyện (Train):    {X_train_scaled.shape}")
print(f"Tập Kiểm chứng (Val):     {X_val_scaled.shape}")
print(f"Tập Kiểm tra (Test):      {X_test_scaled.shape}")
print("-" * 30)

Đã tải dữ liệu thành công.
------------------------------
Tổng số mẫu: 13611
Tập Huấn luyện (Train):    (9527, 9)
Tập Kiểm chứng (Val):     (1361, 9)
Tập Kiểm tra (Test):      (2723, 9)
------------------------------


# Model

## Base

In [6]:
class Classification(ABC):
  @abstractmethod
  def fit(self, X: np.ndarray, y: np.ndarray) -> None:
    """
    Fit the model to the training data.
    """
    pass

  @abstractmethod
  def predict(self, X: np.ndarray) -> np.ndarray:
    """
    Predict the target values for the given input data.
    """
    pass

  @abstractmethod
  def evaluate(self, y_hat: np.ndarray, y: np.ndarray) -> float:
    """
    Evaluate the model on the given input and target data.
    """
    pass

## Logistic Regression

In [ ]:
class LogisticRegression(Classification):
  def __init__(
    self,
    random_state: int = SEED,
    solver: str = 'gradient_descent',
    learning_rate: float = LEARNING_RATE,
    max_iter: int = MAX_ITER,
  ):
    self.random_state = random_state
    self.solver = solver
    self.learning_rate = learning_rate
    self.max_iter = max_iter
    self.coef_ = None
    self.intercept_ = None
    self.history = {'loss': [], 'time': []}
    
  def fit(self, X: np.ndarray, y: np.ndarray) -> None:
    raise NotImplementedError('Not implemented yet')

  def predict(self, X: np.ndarray) -> np.ndarray:
    raise NotImplementedError('Not implemented yet')

  def evaluate(self, y_hat: np.ndarray, y: np.ndarray) -> float:
    raise NotImplementedError('Not implemented yet')

# Evaluation

# Logistic Regression (nhị phân và đa lớp):


Cài đặt Gradient Descent từ đầu cho nhị phân (sigmoid) và đa lớp (softmax).

In [ ]:
import numpy as np
import time

class LogisticRegressionSigmoid:
    def __init__(
        self,
        random_state: int = 42,
        learning_rate: float = 0.01,
        max_iter: int = 1000,
    ):
        self.random_state = random_state
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.coef_ = None
        self.intercept_ = None
        self.history = {'loss': [], 'time': []}

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def _compute_loss(self, y, y_hat):
        # Binary Cross-Entropy Loss
        m = len(y)
        return -(1/m) * np.sum(y * np.log(y_hat + 1e-15) + (1 - y) * np.log(1 - y_hat + 1e-15))

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        
        # Khởi tạo tham số
        self.coef_ = np.zeros(n_features)
        self.intercept_ = 0
        
        start_time = time.time()
        for i in range(self.max_iter):
            # Forward pass
            z = np.dot(X, self.coef_) + self.intercept_
            y_hat = self._sigmoid(z)
            
            # Tính gradient
            dw = (1 / n_samples) * np.dot(X.T, (y_hat - y))
            db = (1 / n_samples) * np.sum(y_hat - y)
            
            # Cập nhật tham số
            self.coef_ -= self.learning_rate * dw
            self.intercept_ -= self.learning_rate * db
            
            # Lưu lịch sử
            if i % 10 == 0:
                self.history['loss'].append(self._compute_loss(y, y_hat))
                self.history['time'].append(time.time() - start_time)

    def predict(self, X: np.ndarray) -> np.ndarray:
        z = np.dot(X, self.coef_) + self.intercept_
        y_hat = self._sigmoid(z)
        return (y_hat >= 0.5).astype(int)

    def evaluate(self, y_hat: np.ndarray, y: np.ndarray) -> float:
        return np.mean(y_hat == y)

In [ ]:
class SoftmaxRegression:
    def __init__(
        self,
        random_state: int = 42,
        learning_rate: float = 0.01,
        max_iter: int = 1000,
    ):
        self.random_state = random_state
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.coef_ = None # Ma trận W (features x classes)
        self.intercept_ = None # Vector b (classes,)
        self.history = {'loss': [], 'time': []}

    def _softmax(self, z):
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True)) # Ổn định số học
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def _one_hot(self, y, k):
        one_hot = np.zeros((len(y), k))
        one_hot[np.arange(len(y)), y] = 1
        return one_hot

    def _compute_loss(self, y_true_oh, y_hat):
        # Multi-class Cross-Entropy Loss
        m = y_true_oh.shape[0]
        return -(1/m) * np.sum(y_true_oh * np.log(y_hat + 1e-15))

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        classes = np.unique(y)
        k = len(classes)
        y_oh = self._one_hot(y, k)
        
        self.coef_ = np.zeros((n_features, k))
        self.intercept_ = np.zeros(k)
        
        start_time = time.time()
        for i in range(self.max_iter):
            # Forward pass
            z = np.dot(X, self.coef_) + self.intercept_
            y_hat = self._softmax(z)
            
            # Gradient
            dw = (1 / n_samples) * np.dot(X.T, (y_hat - y_oh))
            db = (1 / n_samples) * np.sum(y_hat - y_oh, axis=0)
            
            # Update
            self.coef_ -= self.learning_rate * dw
            self.intercept_ -= self.learning_rate * db
            
            if i % 10 == 0:
                self.history['loss'].append(self._compute_loss(y_oh, y_hat))
                self.history['time'].append(time.time() - start_time)

    def predict(self, X: np.ndarray) -> np.ndarray:
        z = np.dot(X, self.coef_) + self.intercept_
        y_hat = self._softmax(z)
        return np.argmax(y_hat, axis=1)

    def evaluate(self, y_hat: np.ndarray, y: np.ndarray) -> float:
        return np.mean(y_hat == y)

Cài đặt Newton–Raphson / IRLS cho bài toán nhị phân và so sánh tốc độ
hội tụ (loss vs. epoch, loss vs. wall-clock time)

In [ ]:
import numpy as np
import time

class LogisticRegressionNewton:
    def __init__(self, solver='newton', max_iter=10, learning_rate=1.0, tol=1e-6):
        self.solver = solver
        self.max_iter = max_iter
        self.lr = learning_rate # Thường = 1.0 cho Newton
        self.tol = tol
        self.coef_ = None
        self.history = {'loss': [], 'time': []}

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def _compute_loss(self, X, y, w):
        a = self._sigmoid(X @ w)
        # Thêm epsilon để tránh log(0)
        return -np.mean(y * np.log(a + 1e-15) + (1 - y) * np.log(1 - a + 1e-15))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Thêm cột bias vào X
        X_bias = np.c_[np.ones(n_samples), X]
        self.coef_ = np.zeros(X_bias.shape[1])
        
        start_total = time.time()
        
        for i in range(self.max_iter):
            iter_start = time.time()
            
            # Forward pass
            z = X_bias @ self.coef_
            a = self._sigmoid(z)
            
            # 1. Tính Gradient
            gradient = X_bias.T @ (a - y) / n_samples
            
            if self.solver == 'newton':
                # 2. Tính Hessian
                # R là ma trận trọng số đường chéo: a_i * (1 - a_i)
                R = a * (1 - a)
                # Hessian = (X.T * R) @ X
                H = (X_bias.T * R) @ X_bias / n_samples
                
                # Thêm Regularization nhỏ để Hessian luôn nghịch đảo được
                H += np.eye(H.shape[0]) * 1e-6
                
                # Cập nhật theo Newton-Raphson: delta = H^-1 * g
                step = np.linalg.solve(H, gradient)
                self.coef_ -= self.lr * step
            else:
                # Gradient Descent cơ bản
                self.coef_ -= self.lr * gradient

            # Lưu lịch sử
            loss = self._compute_loss(X_bias, y, self.coef_)
            self.history['loss'].append(loss)
            self.history['time'].append(time.time() - start_total)
            
            # Kiểm tra hội tụ (Early Stopping)
            if i > 0 and abs(self.history['loss'][-2] - loss) < self.tol:
                break

Cho bài toán đa lớp: so sánh ba chiến lược one-vs-rest, one-vs-one, và
softmax trực tiếp.

### Bảng so sánh các chiến lược phân lớp đa lớp

| Tiêu chí | One-vs-Rest (OvR) | One-vs-One (OvO) | Softmax Regression (Multinomial) |
| :--- | :--- | :--- | :--- |
| **Cơ chế hoạt động** | Huấn luyện $K$ bộ phân lớp nhị phân. Mỗi bộ phân biệt 1 lớp với tất cả các lớp còn lại. | Huấn luyện $K(K-1)/2$ bộ phân lớp. Mỗi bộ phân biệt giữa một cặp lớp $(i, j)$. | Huấn luyện **một** mô hình duy nhất với hàm kích hoạt Softmax để tính xác suất cho tất cả các lớp. |
| **Số lượng mô hình** | $K$ (Ví dụ: 3 lớp $\rightarrow$ 3 mô hình) | $K(K-1)/2$ (Ví dụ: 3 lớp $\rightarrow$ 3 mô hình; 10 lớp $\rightarrow$ 45 mô hình) | **1** (Mô hình hợp nhất) |
| **Độ phức tạp huấn luyện** | Thấp. Mỗi mô hình chạy trên toàn bộ tập dữ liệu. | Cao về số lượng mô hình, nhưng mỗi mô hình chỉ chạy trên dữ liệu của 2 lớp (nhanh hơn). | Trung bình. Cần tính toán ma trận trọng số lớn hơn và dùng hàm mất mát Cross-Entropy. |
| **Cách dự đoán** | Chọn lớp có xác suất (confidence score) cao nhất từ $K$ mô hình. | Bỏ phiếu (Voting). Lớp nào thắng nhiều cặp đấu nhất sẽ được chọn. | Chọn lớp có giá trị xác suất đầu ra lớn nhất (tổng các xác suất bằng 1). |
| **Ưu điểm** | Dễ cài đặt, trực quan, tốn ít tài nguyên bộ nhớ hơn OvO khi số lớp lớn. | Tránh được vấn đề mất cân bằng dữ liệu; hoạt động tốt với SVM hoặc các thuật toán nhạy cảm với kích thước dữ liệu. | Chính xác nhất về mặt toán học vì tối ưu hóa tất cả các lớp cùng lúc; không bị trùng lặp ranh giới. |
| **Nhược điểm** | Dễ bị mất cân bằng dữ liệu (lớp "Rest" thường chiếm đa số); các vùng ranh giới có thể bị chồng lấn. | Số lượng mô hình tăng rất nhanh khi số lớp tăng (bùng nổ tổ hợp). | Yêu cầu thay đổi cấu trúc hàm Loss và đạo hàm (phức tạp hơn khi tự code từ đầu). |

---

Trình bày đầy đủ đạo hàm Hessian, Jacobian của softmax.

Để triển khai **Softmax Regression** (còn gọi là Multinomial Logistic Regression) một cách chuyên sâu, việc hiểu các thành phần vi phân như Jacobian và Hessian là cực kỳ quan trọng để tối ưu hóa bằng các thuật toán bậc 1 (Gradient Descent) hoặc bậc 2 (Newton's Method).

Giả sử ta có vector đầu vào $z = [z_1, z_2, ..., z_K]^T$, hàm Softmax $\sigma(z)$ trả về một vector $a$ cùng chiều, với phần tử thứ $i$ là:
$$a_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

---

### 1. Ma trận Jacobian của Softmax
Ma trận Jacobian $J$ chứa tất cả các đạo hàm riêng bậc nhất của các đầu ra đối với các đầu vào. Vì mỗi $a_i$ phụ thuộc vào tất cả các $z_j$, ma trận này có kích thước $K \times K$.

Phần tử tại hàng $i$, cột $j$ của ma trận Jacobian là $\frac{\partial a_i}{\partial z_j}$:

*   **Trường hợp $i = j$ (Đạo hàm riêng với chính nó):**
    $$\frac{\partial a_i}{\partial z_i} = a_i(1 - a_i)$$
*   **Trường hợp $i \neq j$ (Đạo hàm riêng chéo):**
    $$\frac{\partial a_i}{\partial z_j} = -a_i a_j$$

**Dạng ma trận tổng quát:**
$$J = \text{diag}(a) - aa^T$$
Trong đó $\text{diag}(a)$ là ma trận đường chéo với các phần tử của vector $a$.

---

### 2. Đạo hàm của hàm mất mát (Gradient)
Trong bài toán phân lớp, Softmax luôn đi kèm với hàm mất mát **Cross-Entropy** ($L$). Giả sử $y$ là vector nhãn (one-hot vector). Gradient của $L$ đối với đầu vào $z$ có dạng cực kỳ đơn giản:

$$\nabla_z L = a - y$$

Đây chính là lý do vì sao cặp bài trùng Softmax và Cross-Entropy lại phổ biến đến vậy: lỗi (error) đơn giản là sự chênh lệch giữa xác suất dự đoán và nhãn thực tế.



---

### 3. Ma trận Hessian của Softmax
Ma trận Hessian $H$ là đạo hàm bậc hai của hàm mất mát $L$ đối với đầu vào $z$. Nó cho biết độ cong của bề mặt lỗi, giúp các thuật toán như Newton's Method hội tụ nhanh hơn.

Từ kết quả Gradient $\nabla_z L = a - y$, ta lấy đạo hàm một lần nữa theo $z$:
$$H = \frac{\partial^2 L}{\partial z^2} = \frac{\partial (a - y)}{\partial z}$$

Vì $y$ là hằng số đối với $z$, nên Hessian của hàm mất mát chính là **Jacobian của Softmax**:
$$H = \frac{\partial a}{\partial z} = \text{diag}(a) - aa^T$$

#### Đặc điểm của Hessian trong Softmax:
1.  **Kích thước:** $K \times K$.
2.  **Tính đối xứng:** $H_{ij} = H_{ji} = -a_i a_j$.
3.  **Bán xác định dương (Positive Semi-definite):** Điều này đảm bảo hàm mất mát Cross-Entropy là hàm lồi (convex), nghĩa là nếu ta tìm được cực tiểu thì đó chắc chắn là cực tiểu toàn cục.
4.  **Suy biến:** Ma trận này không nghịch đảo được vì tổng các hàng (hoặc cột) luôn bằng 0 (do $\sum a_i = 1$). Trong thực tế, người ta thường thêm một hạng tử điều hòa (regularization) để làm Hessian nghịch đảo được.

---

### Tóm tắt công thức triển khai

| Thành phần | Công thức | Ý nghĩa |
| :--- | :--- | :--- |
| **Output** | $a = \text{softmax}(z)$ | Dự đoán xác suất. |
| **Jacobian** | $J_{ij} = a_i(\delta_{ij} - a_j)$ | Tốc độ thay đổi của xác suất. |
| **Gradient** | $g = a - y$ | Hướng cập nhật trọng số (bậc 1). |
| **Hessian** | $H = \text{diag}(a) - aa^T$ | Độ cong của hàm lỗi (bậc 2). |

**Bạn có muốn tôi hỗ trợ viết hàm tính toán Hessian này bằng thư viện NumPy để áp dụng vào phương pháp tối ưu Newton không?**